# Advanced Macroeconomic Analysis of Housing Demand
This notebook provides a structural framework to evaluate the elasticity of housing demand, allowing a portfolio manager to determine if a market is in a regime of expansion or contraction.

## 1. Structural Decomposition of Demand
The agent's participation in the housing market is defined by the condition where the marginal utility of housing services exceeds the user cost of capital ($uc_t$):

$uc_t = i_t + \tau_t + \delta - \mathbb{E}_t[\pi^h_{t+1}]$

Where:
- $i_t$: Nominal mortgage rate
- $\tau_t$: Property taxes/insurance
- $\delta$: Depreciation
- $\mathbb{E}_t[\pi^h_{t+1}]$: Expected capital appreciation

In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

%load_ext sql
%sql duckdb:///:memory:

Connecting to 'duckdb:///:memory:'

In [2]:
%%sql
INSTALL httpfs;
LOAD httpfs;

Running query in 'duckdb:///:memory:'

Success


In [16]:
%%sql housing_data <<
WITH raw_data AS (
    SELECT *
    FROM read_csv(
        'https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/prc_hpi_q?format=SDMX-CSV&lang=en',
        union_by_name = true,
        null_padding = true
    )
)
SELECT
    TIME_PERIOD AS quarter,
    CASE geo
        WHEN 'IT' THEN 'Italy'
        WHEN 'FR' THEN 'France'
        WHEN 'DE' THEN 'Germany'
        WHEN 'ES' THEN 'Spain'
    END AS sovereign_state,
    CAST(OBS_VALUE AS DOUBLE) AS hpi_index
FROM raw_data
WHERE geo IN ('IT', 'FR', 'DE', 'ES')
  AND OBS_VALUE IS NOT NULL
ORDER BY quarter, sovereign_state;

Running query in 'duckdb:///:memory:'

In [18]:
import pandas as pd
import statsmodels.api as sm

data = housing_data.DataFrame().copy()

data['quarter'] = pd.PeriodIndex(data['quarter'], freq='Q')
data['t'] = range(len(data))

X = sm.add_constant(data['t'])
y = data['hpi_index']

model = sm.OLS(y, X).fit()

print(model.summary())

                            OLS Regression Results                            
Dep. Variable:              hpi_index   R-squared:                       0.010
Model:                            OLS   Adj. R-squared:                  0.009
Method:                 Least Squares   F-statistic:                     35.88
Date:                Mon, 15 Jun 2026   Prob (F-statistic):           2.30e-09
Time:                        23:10:24   Log-Likelihood:                -20234.
No. Observations:                3696   AIC:                         4.047e+04
Df Residuals:                    3694   BIC:                         4.048e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         47.5169      1.899     25.023      0.0

## 2. Portfolio Management Intelligence
### Elasticity Interpretation
- If the elasticity coefficient (the slope) is $<-1$, the market is **hyper-sensitive** to interest rate changes.
- A widening gap between the trend of transaction volume and the historical mean indicates a breakdown in the 'buying' mechanism, likely requiring a shift to defensive hedging strategies.